# 📒 Cuadernillo Módulo 3 — Machine Learning con scikit-learn
**Diplomado Superior en Redes Neuronales Artificiales y Deep Learning**  
Diana Blanco | Referencia rápida: scikit-learn, clasificación, regresión, clustering

> 📖 **¿Para qué sirve este archivo?** → Consulta rápida de scikit-learn. Tablas con  
> modelos, métricas, parámetros y cuándo usar cada uno. Sin código ejecutable, solo  
> referencia teórica para tenerla abierta mientras trabajas.

---
> Todo lo que necesitas recordar de scikit-learn para construir modelos de ML.

---
## 📑 Índice
| Sec | Contenido |
|-----|-----------|
| **1** | Flujo de ML — pipeline de 10 pasos |
| **2** | Modelos de clasificación — Perceptron, LogisticRegression, DecisionTree, SVC |
| **3** | Modelos de regresión — LinearRegression, DecisionTreeRegressor, SVR |
| **4** | Clustering — KMeans, método del codo |
| **5** | Métricas — accuracy, precision, recall, f1, MSE, RMSE, MAE, R² |
| **6** | Preprocesamiento — train_test_split, escaladores, codificadores |
| **7** | Funciones de activación — sigmoide, tanh, ReLU, escalón |
| **8** | Compuertas lógicas — AND, OR, XOR |
| **9** | Errores comunes — troubleshooting

---
## 1. 🔄 Flujo de Machine Learning (Pipeline de 10 pasos)

```
 1. Cargar       → pd.read_csv('datos.csv')  — importar datos
 2. EDA          → df.info(), describe(), isnull(), corr()  — explorar
 3. Features     → elegir X (variables predictoras) y y (objetivo)
 4. Dividir      → train_test_split(X, y, test_size=0.2, random_state=42)
 5. Escalar      → StandardScaler() o MinMaxScaler() — solo en X
 6. Modelo       → elegir algoritmo (clasificación / regresión / clustering)
 7. Entrenar     → modelo.fit(X_train, y_train)
 8. Predecir     → y_pred = modelo.predict(X_test)
 9. Evaluar      → métricas (accuracy, MSE, etc.)
10. Comparar     → probar varios modelos y elegir el mejor
```

> ⚠️ **Regla de oro:** Escalar SIEMPRE *después* de dividir, nunca antes (data leakage).

---
## 2. 🏷️ Modelos de Clasificación

| Modelo | Clase | Parámetros clave | Ideal para |
|--------|-------|-------------------|------------|
| **Perceptrón** | `sklearn.linear_model.Perceptron` | `max_iter=1000`, `eta0=0.1`, `random_state=42` | Datos linealmente separables |
| **Regresión Logística** | `sklearn.linear_model.LogisticRegression` | `C=1.0`, `solver='lbfgs'`, `max_iter=200` | Probabilidades de clase, binaria/multiclase |
| **Árbol de Decisión** | `sklearn.tree.DecisionTreeClassifier` | `max_depth=5`, `criterion='gini'`, `min_samples_split=2` | Datos interpretables, no requiere escalado |
| **SVM** | `sklearn.svm.SVC` | `C=1.0`, `kernel='rbf'`, `gamma='scale'` | Fronteras complejas, alta dimensionalidad |

```python
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

modelos = {
    'Perceptron': Perceptron(max_iter=1000, random_state=42),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=200, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'SVM': SVC(kernel='rbf', C=1.0, random_state=42)
}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    print(f'{nombre}: {modelo.score(X_test, y_test):.3f}')
```

---
## 3. 📈 Modelos de Regresión

| Modelo | Clase | Parámetros clave | Ideal para |
|--------|-------|-------------------|------------|
| **Regresión Lineal** | `sklearn.linear_model.LinearRegression` | `fit_intercept=True` | Relaciones lineales simples/múltiples |
| **Árbol de Regresión** | `sklearn.tree.DecisionTreeRegressor` | `max_depth=5`, `min_samples_split=2` | Datos no lineales, interpretable |
| **SVR** | `sklearn.svm.SVR` | `kernel='rbf'`, `C=1.0`, `epsilon=0.1` | Relaciones complejas, pocos datos |

```python
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

modelos = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'SVR': SVR(kernel='rbf', C=100, gamma='scale')
}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f'{nombre}: MSE={mse:.2f}, R²={r2:.3f}')
```

> 💡 Si quieres regresión polinomial, usa `PolynomialFeatures(degree=2)` + `LinearRegression`.

---
## 4. 🧩 Clustering — KMeans y método del codo

```python
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Escalar siempre antes de clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Método del codo para encontrar k óptimo
inercia = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inercia.append(km.inertia_)

# Graficar el codo
plt.plot(range(1, 11), inercia, marker='o')
plt.xlabel('k (número de clusters)')
plt.ylabel('Inercia (WCSS)')
plt.title('Método del Codo')
plt.grid(alpha=0.3)
plt.show()

# Entrenar con k óptimo
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
kmeans.fit(X_scaled)
labels = kmeans.labels_
centros = kmeans.cluster_centers_
```

> La **inercia** es la suma de distancias al cuadrado de cada punto a su centroide (WCSS).  
> El codo está donde la inercia deja de disminuir drásticamente.

---
## 5. 📊 Métricas de Evaluación

### Clasificación
```python
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

accuracy  = accuracy_score(y_test, y_pred)                    # (VP+VN) / total
precision = precision_score(y_test, y_pred, average='weighted')  # VP / (VP+FP)
recall    = recall_score(y_test, y_pred, average='weighted')     # VP / (VP+FN)
f1        = f1_score(y_test, y_pred, average='weighted')         # 2·P·R / (P+R)

cm = confusion_matrix(y_test, y_pred)                        # matriz de confusión
print(classification_report(y_test, y_pred))                  # reporte completo
```

| Métrica | Fórmula | Qué mide |
|---------|---------|----------|
| Accuracy | (VP+VN)/(VP+VN+FP+FN) | % de aciertos total |
| Precision | VP/(VP+FP) | De los que predijo positivo, cuántos lo eran |
| Recall | VP/(VP+FN) | De los positivos reales, cuántos atrapó |
| F1-Score | 2·P·R/(P+R) | Balance armónico entre precisión y recall |

### Regresión
```python
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse  = mean_squared_error(y_test, y_pred)        # Error cuadrático medio
rmse = np.sqrt(mse)                              # Raíz del MSE (misma unidad que y)
mae  = mean_absolute_error(y_test, y_pred)       # Error absoluto medio
r2   = r2_score(y_test, y_pred)                  # Coeficiente de determinación
```

| Métrica | Fórmula | Rango ideal |
|---------|---------|-------------|
| MSE | (1/n)·Σ(yᵢ−ŷᵢ)² | → 0 (óptimo) |
| RMSE | √MSE | → 0 (óptimo) |
| MAE | (1/n)·Σ|yᵢ−ŷᵢ| | → 0 (óptimo) |
| R² | 1 − Σ(yᵢ−ŷᵢ)²/Σ(yᵢ−ȳ)² | → 1 (óptimo) |

---
## 6. 🧹 Preprocesamiento

### train_test_split — dividir datos
```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify para clasificación
)
```

### Escaladores
| Escalador | Fórmula | Cuándo usarlo |
|-----------|---------|---------------|
| **StandardScaler** | z = (x − μ) / σ | Distribución normal/gaussiana (SVM, Perceptrón, Reg Logística) |
| **MinMaxScaler** | x' = (x − min) / (max − min) | Rangos acotados [0,1], distribuciones uniformes |

```python
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Estandarización (media=0, std=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)          # solo transform, no fit

# Normalización (rango [0,1])
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
```

### Codificadores de variables categóricas
| Codificador | Uso | Ejemplo |
|-------------|-----|---------|
| **LabelEncoder** | y objetivo (etiquetas a enteros) | `['gato','perro']` → `[0, 1]` |
| **OneHotEncoder** | X categóricas (sin orden) | `['rojo','verde','azul']` → `[1,0,0] [0,1,0] [0,0,1]` |

```python
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Para la variable objetivo (y)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Para características categóricas (X)
ohe = OneHotEncoder(sparse_output=False)
X_ohe = ohe.fit_transform(X_cat)
```

> ⚠️ **Nunca** escales la variable objetivo `y` en regresión.  
> ⚠️ Usa `sparse_output=False` en OneHotEncoder para obtener array denso.

---
## 7. ⚡ Funciones de Activación

| Función | Fórmula | Rango | Gráfica conceptual | Usos |
|---------|---------|-------|--------------------|------|
| **Escalón** (Step) | f(x) = 1 si x ≥ 0, 0 si x < 0 | {0, 1} | __|￣ | Perceptrón clásico, MPNeuron |
| **Sigmoide** | σ(x) = 1/(1+e⁻ˣ) | (0, 1) | ∼ (curva S suave) | Regresión logística, probabilidades |
| **Tanh** | tanh(x) = (eˣ−e⁻ˣ)/(eˣ+e⁻ˣ) | (−1, 1) | ∼ (curva S centrada) | Capas ocultas de redes |
| **ReLU** | f(x) = max(0, x) | [0, ∞) | _|╱ (doble recta) | Redes profundas (evita saturación) |

```python
import numpy as np

def escalon(x):     return np.where(x >= 0, 1, 0)
def sigmoide(x):    return 1 / (1 + np.exp(-x))
def tanh(x):        return np.tanh(x)
def relu(x):        return np.maximum(0, x)

# Visualización rápida
import matplotlib.pyplot as plt
x = np.linspace(-5, 5, 100)
plt.plot(x, escalon(x), label='Step')
plt.plot(x, sigmoide(x), label='Sigmoide')
plt.plot(x, tanh(x), label='Tanh')
plt.plot(x, relu(x), label='ReLU')
plt.axhline(0, color='gray', alpha=0.3)
plt.axvline(0, color='gray', alpha=0.3)
plt.legend()
plt.grid(alpha=0.3)
plt.title('Funciones de Activación')
plt.show()
```

---
## 8. 🔲 Compuertas Lógicas

### Tablas de verdad
```
  AND                OR                 XOR
A B | Q             A B | Q            A B | Q
----|--             ----|--            ----|--
0 0 | 0             0 0 | 0            0 0 | 0
0 1 | 0             0 1 | 1            0 1 | 1
1 0 | 0             1 0 | 1            1 0 | 1
1 1 | 1             1 1 | 1            1 1 | 0
```

### ¿Qué perceptrón puede aprender cada una?
| Compuerta | Linealmente separable | Perceptrón simple | Requiere MLP |
|-----------|-----------------------|-------------------|--------------|
| AND | ✅ Sí | ✅ Sí | ❌ No |
| OR | ✅ Sí | ✅ Sí | ❌ No |
| XOR | ❌ No | ❌ No | ✅ Sí |

### Pesos conocidos (Perceptrón)
| Compuerta | w₁ | w₂ | b |
|-----------|----|----|---|
| AND | 0.5 | 0.5 | −0.7 |
| OR | 0.5 | 0.5 | −0.2 |
| NAND | −0.5 | −0.5 | 0.7 |

```python
def perceptron(x1, x2, w1, w2, b):
    suma = w1*x1 + w2*x2 + b
    return 1 if suma >= 0 else 0

# Probar AND
print(perceptron(0,0, 0.5,0.5,-0.7))  # → 0
print(perceptron(1,1, 0.5,0.5,-0.7))  # → 1
```

---
## 9. 🐛 Errores Comunes

| Error | Causa | Solución |
|-------|-------|----------|
| `ValueError: Input contains NaN, infinity or a value too large` | Datos con nulos o infinitos | `df.dropna()` o `df.fillna(valor)` antes de entrenar |
| `ValueError: could not convert string to float` | Variables categóricas en X sin codificar | Aplicar `OneHotEncoder` o `LabelEncoder` |
| `ConvergenceWarning` | Modelo no convergió (Perceptrón, LogReg) | Aumentar `max_iter`, escalar datos, reducir `tol` |
| Accuracy perfecta (1.0) en test | Data leakage o sobreajuste extremo | Verificar que no se escaló antes de dividir, revisar `test_size` |
| `RandomState` no reproducible | Falta `random_state` | `modelo = Modelo(random_state=42)` |
| SVC muy lento | Dataset grande + kernel rbf | Usar `LinearSVC` o `SGDClassifier` |
| KMeans: diferentes clusters cada vez | `n_init` bajo o falta `random_state` | `KMeans(n_init=10, random_state=42)` |
| `AttributeError: 'numpy.ndarray' object has no attribute 'columns'` | Usaste .fit_transform y perdiste nombres | Guardar scaler/encoder y convertir a DataFrame con `pd.DataFrame(X_scaled, columns=...).` |
| Matriz de confusión no cuadrada | Desbalance de clases o error en etiquetas | Verificar `unique(y_test)` y `unique(y_pred)` |
| R² negativo | Modelo peor que la media | Revisar si datos son lineales, probar otro modelo, escalar X |